# **Social Action Recognition in Mice**

## APPROACH

- Train a classifier to differentiate between self-action and actions toward other mice
- Train a classifier that identifies self-action (rear)
- Train a multiclass classifier that identifies actions towards other mice (approach, attack, avoid, chase, chaseattack, submit)
- At first use AdaptableSnail Lab data only, later try to learn from other labs too

## **SETUP**

In [36]:
!pip install --upgrade scikit-learn

### IMPORTS, SETTINGS AND GLOBALS

In [22]:
import pathlib
import pandas as pd
import numpy as np
import ast
import sklearn
import json
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

pd.options.display.max_rows = 100
sklearn.set_config(transform_output="pandas")

COMPETITION_NAME = "MABe-mouse-behavior-detection"
#INPUT_DIRECTORY = pathlib.Path("/kaggle/input/") / COMPETITION_NAME
INPUT_DIRECTORY = pathlib.Path().cwd() / "kaggle" / "input" / COMPETITION_NAME
#OUTPUT_DIRECTORY = pathlib.Path("/kaggle/working")
OUTPUT_DIRECTORY = pathlib.Path().cwd() / "kaggle" / "working"

LABS_CONSIDERED = ["AdaptableSnail"]

PROBA_THR = 0.15
MIN_FRAMES = 2
N_CV_SPLITS = 7
RANDOM_STATE = 42

### CUSTOM METRIC

In [23]:
"""F Beta customized for the data format of the MABe challenge."""

import json

from collections import defaultdict

import pandas as pd
import polars as pl


class HostVisibleError(Exception):
    pass


def single_lab_f1(lab_solution: pl.DataFrame, lab_submission: pl.DataFrame, beta: float = 1) -> float:
    label_frames: defaultdict[str, set[int]] = defaultdict(set)
    prediction_frames: defaultdict[str, set[int]] = defaultdict(set)

    for row in lab_solution.to_dicts():
        label_frames[row['label_key']].update(range(row['start_frame'], row['stop_frame']))

    for video in lab_solution['video_id'].unique():
        active_labels: str = lab_solution.filter(pl.col('video_id') == video)['behaviors_labeled'].first()  # ty: ignore
        active_labels: set[str] = set(json.loads(active_labels))
        predicted_mouse_pairs: defaultdict[str, set[int]] = defaultdict(set)

        for row in lab_submission.filter(pl.col('video_id') == video).to_dicts():
            # Since the labels are sparse, we can't evaluate prediction keys not in the active labels.
            if ','.join([str(row['agent_id']), str(row['target_id']), row['action']]) not in active_labels:
                continue

            new_frames = set(range(row['start_frame'], row['stop_frame']))
            # Ignore truly redundant predictions.
            new_frames = new_frames.difference(prediction_frames[row['prediction_key']])
            prediction_pair = ','.join([str(row['agent_id']), str(row['target_id'])])
            if predicted_mouse_pairs[prediction_pair].intersection(new_frames):
                # A single agent can have multiple targets per frame (ex: evading all other mice) but only one action per target per frame.
                raise HostVisibleError('Multiple predictions for the same frame from one agent/target pair')
            prediction_frames[row['prediction_key']].update(new_frames)
            predicted_mouse_pairs[prediction_pair].update(new_frames)

    tps = defaultdict(int)
    fns = defaultdict(int)
    fps = defaultdict(int)
    for key, pred_frames in prediction_frames.items():
        action = key.split('_')[-1]
        matched_label_frames = label_frames[key]
        tps[action] += len(pred_frames.intersection(matched_label_frames))
        fns[action] += len(matched_label_frames.difference(pred_frames))
        fps[action] += len(pred_frames.difference(matched_label_frames))

    distinct_actions = set()
    for key, frames in label_frames.items():
        action = key.split('_')[-1]
        distinct_actions.add(action)
        if key not in prediction_frames:
            fns[action] += len(frames)

    action_f1s = []
    for action in distinct_actions:
        if tps[action] + fns[action] + fps[action] == 0:
            action_f1s.append(0)
        else:
            action_f1s.append((1 + beta**2) * tps[action] / ((1 + beta**2) * tps[action] + beta**2 * fns[action] + fps[action]))
    return sum(action_f1s) / len(action_f1s)


def mouse_fbeta(solution: pd.DataFrame, submission: pd.DataFrame, beta: float = 1) -> float:
    if len(solution) == 0 or len(submission) == 0:
        raise ValueError('Missing solution or submission data')

    expected_cols = ['video_id', 'agent_id', 'target_id', 'action', 'start_frame', 'stop_frame']

    for col in expected_cols:
        if col not in solution.columns:
            raise ValueError(f'Solution is missing column {col}')
        if col not in submission.columns:
            raise ValueError(f'Submission is missing column {col}')

    solution: pl.DataFrame = pl.DataFrame(solution)
    submission: pl.DataFrame = pl.DataFrame(submission)
    assert (solution['start_frame'] <= solution['stop_frame']).all()
    assert (submission['start_frame'] <= submission['stop_frame']).all()
    solution_videos = set(solution['video_id'].unique())
    # Need to align based on video IDs as we can't rely on the row IDs for handling public/private splits.
    submission = submission.filter(pl.col('video_id').is_in(solution_videos))

    solution = solution.with_columns(
        pl.concat_str(
            [
                pl.col('video_id').cast(pl.Utf8),
                pl.col('agent_id').cast(pl.Utf8),
                pl.col('target_id').cast(pl.Utf8),
                pl.col('action'),
            ],
            separator='_',
        ).alias('label_key'),
    )
    submission = submission.with_columns(
        pl.concat_str(
            [
                pl.col('video_id').cast(pl.Utf8),
                pl.col('agent_id').cast(pl.Utf8),
                pl.col('target_id').cast(pl.Utf8),
                pl.col('action'),
            ],
            separator='_',
        ).alias('prediction_key'),
    )

    lab_scores = []
    for lab in solution['lab_id'].unique():
        lab_solution = solution.filter(pl.col('lab_id') == lab).clone()
        lab_videos = set(lab_solution['video_id'].unique())
        lab_submission = submission.filter(pl.col('video_id').is_in(lab_videos)).clone()
        lab_scores.append(single_lab_f1(lab_solution, lab_submission, beta=beta))

    return sum(lab_scores) / len(lab_scores)


def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str, beta: float = 1) -> float:
    """
    F1 score for the MABe Challenge
    """
    solution = solution.drop(row_id_column_name, axis='columns', errors='ignore')
    submission = submission.drop(row_id_column_name, axis='columns', errors='ignore')
    return mouse_fbeta(solution, submission, beta=beta)

### HELPER FUNCTIONS

In [24]:
def ingest_tracking_data(split):
    assert split in ["train", "test"]
    ##################################################
    ##########   tracking data ingestion    ##########
    ##################################################
    tracking_df = pd.DataFrame()
    for lab in LABS_CONSIDERED:
        tracking_directory = INPUT_DIRECTORY / f"{split}_tracking" / lab
        for tracking_file in tracking_directory.glob("*.parquet"):
            print(f"reading {tracking_file}")
            tmp = pd.read_parquet(tracking_file)
            tmp["video_id"] = str(tracking_file).split("/")[-1].replace(".parquet","")
            tracking_df = pd.concat([tracking_df, tmp])
    print("\ntracking data ingestion complete:")
    print(f"rows: {tracking_df.shape[0]}")
    print(f"cols: {tracking_df.shape[1]}")
    print(f"\n{tracking_df.sample(n=10)}")
    
    ############################################################
    ########## estimate body pose per frame and mouse ##########
    ############################################################
    # preprocess tracking data
    tracking_df["bodyregion"] = tracking_df["bodypart"].map(bodypart_to_region)
    tracking_df = tracking_df.dropna(subset=["bodyregion"])
    region_df = tracking_df.groupby(["video_id", "video_frame", "mouse_id", "bodyregion"], as_index=False)[["x", "y"]].mean()
    pivot_df = region_df.pivot_table(
        index=["video_id", "video_frame", "mouse_id"],
        columns="bodyregion",
        values=["x", "y"]
    )
    
    # flatten MultiIndex columns
    pivot_df.columns = [f"{coord}_{region}" for coord, region in pivot_df.columns]
    pivot_df = pivot_df.reset_index()
    
    # vector from torso to head
    pivot_df["x_dir"] = pivot_df["x_head"] - pivot_df["x_torso"]
    pivot_df["y_dir"] = pivot_df["y_head"] - pivot_df["y_torso"]
    norm = np.sqrt(pivot_df["x_dir"]**2 + pivot_df["y_dir"]**2)
    pivot_df["x_dir"] = pivot_df["x_dir"] / norm
    pivot_df["y_dir"] = pivot_df["y_dir"] / norm
    
    tracking_df = pivot_df[["video_id", "video_frame", "mouse_id", "x_torso", "y_torso", "x_dir", "y_dir"]].rename(columns={"x_torso":"x_pos", "y_torso":"y_pos"})
    tracking_df["x_pos"] = tracking_df["x_pos"] / metadata.loc[tracking_df["video_id"], "pix_per_cm_approx"]
    tracking_df["y_pos"] = tracking_df["y_pos"] / metadata.loc[tracking_df["video_id"], "pix_per_cm_approx"]
    print(f"\nextracting position and direction from tracking data complete:\n{tracking_df.head()}")
    
    ##################################################
    ##########    convert to wide format    ##########
    ##################################################
    # pivot to get one row per frame, four columns per mouse
    tracking_df["mouse_num"] = tracking_df["mouse_id"].astype(str).str.extract(r"(\d+)").astype(int)
    wide = tracking_df.pivot_table(
        index=["video_id", "video_frame"],
        columns="mouse_num",
        values=["x_pos","y_pos","x_dir","y_dir"],
        aggfunc="first"
    )
    
    # flatten MultiIndex columns like ("x_pos", 1) -> "x_pos_1"
    wide.columns = [f"{feat}_{int(m)}" for feat, m in wide.columns]
    wide = wide.reset_index()
    
    # ensure columns for all 4 mice in order
    wanted = [f"{f}_{m}" for m in [1,2,3,4] for f in ["x_pos","y_pos","x_dir","y_dir"]]
    for c in wanted:
        if c not in wide.columns:
            wide[c] = np.nan
    wide = wide[["video_id", "video_frame"] + wanted]
    wide = wide.dropna()
    
    tracking_df = wide.copy()
    print(f"\nconversion to wide format complete:\n{tracking_df.head()}")
    return tracking_df

def ingest_annotation_data(split="train"):
    assert split in ["train"]
    ##################################################
    ##########  annotation data ingestion   ##########
    ##################################################
    annotation_df = pd.DataFrame()
    for lab in LABS_CONSIDERED:
        annotation_directory = INPUT_DIRECTORY / f"{split}_annotation" / lab
        for annotation_file in annotation_directory.glob("*.parquet"):
            print(f"reading {annotation_file}")
            tmp = pd.read_parquet(annotation_file)
            tmp["video_id"] = str(annotation_file).split("/")[-1].replace(".parquet","")
            annotation_df = pd.concat([annotation_df, tmp])
            del tmp
    
    print("\nannotation data ingestion complete:")
    print(f"rows: {annotation_df.shape[0]}")
    print(f"cols: {annotation_df.shape[1]}")
    print(f"\n{annotation_df.sample(n=10)}")
    
    ##################################################
    ######### explode df to per-frame basis ##########
    ##################################################
    tmp = annotation_df.copy()
    tmp["video_frame"] = [np.arange(s, e + 1) for s, e in zip(tmp.start_frame, tmp.stop_frame)]
    exploded = tmp.explode("video_frame", ignore_index=True)
    annotation_df = exploded[["video_id", "video_frame", "agent_id", "target_id", "action"]]
    del tmp
    del exploded
    
    print("\nafter exploding on frame-level:")
    print(f"rows: {annotation_df.shape[0]}")
    print(f"cols: {annotation_df.shape[1]}")
    print(f"\n{annotation_df.sample(n=10)}")
    return annotation_df

def make_slot_maps(ego: int):
    """Return (id to slot, slot toid) for this ego. Slots are ['self','A','B','C'], ego is an integer between 1 and 4."""
    others = sorted({1,2,3,4} - {ego})
    id2slot = {ego: "self", others[0]: "A", others[1]: "B", others[2]: "C"}
    slot2id = {v: k for k, v in id2slot.items()}
    return id2slot, slot2id

def tracking_to_ego_df(tracking_df: pd.DataFrame, ego: int) -> pd.DataFrame:
    id2slot, _ = make_slot_maps(ego)
    out = pd.DataFrame(index=tracking_df.index)
    out[["video_id", "video_frame"]] = tracking_df[["video_id", "video_frame"]]
    out["agent_id"] = ego

    # self as-is
    out["x_pos_self"] = tracking_df[f"x_pos_{ego}"]
    out["y_pos_self"] = tracking_df[f"y_pos_{ego}"]
    out["x_dir_self"] = tracking_df[f"x_dir_{ego}"]
    out["y_dir_self"] = tracking_df[f"y_dir_{ego}"]

    # others as (self - other)
    for mid, slot in id2slot.items():
        if slot == "self":
            continue
        out[f"x_pos_{slot}"] = tracking_df[f"x_pos_{ego}"] - tracking_df[f"x_pos_{mid}"]
        out[f"y_pos_{slot}"] = tracking_df[f"y_pos_{ego}"] - tracking_df[f"y_pos_{mid}"]
        out[f"x_dir_{slot}"] = tracking_df[f"x_dir_{ego}"] - tracking_df[f"x_dir_{mid}"]
        out[f"y_dir_{slot}"] = tracking_df[f"y_dir_{ego}"] - tracking_df[f"y_dir_{mid}"]

    return out

def annotation_to_ego_df(annotation_df: pd.DataFrame, ego: int) -> pd.DataFrame:
    """
    Keep only annotations where this ego is the agent.
    Create a joint ego-centric label: 'action_slot', where slot is self/A/B/C
    derived from the ego's perspective of target_id.
    """
    id2slot, _ = make_slot_maps(ego)

    # keep only rows where the acting mouse is the ego
    out = annotation_df.loc[annotation_df["agent_id"] == ego].copy()

    # map absolute target_id -> ego slot
    out["target_slot"] = out["target_id"].map(id2slot)

    # build joint label; if mapping is missing (shouldn't with ids 1..4), set to no_action
    out["label"] = np.where(
        out["target_slot"].notna(),
        out["action"] + "_" + out["target_slot"],
        "no_action",
    )

    # return lean columns (keep agent_id=ego for clarity)
    return out[["video_id", "video_frame", "agent_id", "label"]]

def transform_ego_predictions(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if len(df) == 0:
        return pd.DataFrame(columns=["video_id", "agent_id", "target_id", "action", "start_frame", "stop_frame"])
    # 1) Split "action" into base action and slot
    action_split = out["action"].str.rsplit("_", n=1, expand=True)
    out["base_action"] = action_split[0]
    out["slot"] = action_split[1]

    # 2) Map slot back to target_id using agent_id
    def slot_to_target(agent_id, slot):
        _, slot2id = make_slot_maps(int(agent_id))
        return slot2id[slot]

    out["target_id"] = [slot_to_target(a, s) for a, s in zip(out["agent_id"], out["slot"])]

    # 3) Rename base_action back to action and clean up
    out["action"] = out["base_action"]
    out = out.drop(columns=["base_action", "slot"])

    out["agent_id"] = "mouse" + out["agent_id"].astype(str)
    out["target_id"] = "mouse" + out["target_id"].astype(str)
    out["target_id"] = np.where(out["agent_id"] == out["target_id"], "self", out["target_id"])
    return out[["video_id", "agent_id", "target_id", "action", "start_frame", "stop_frame"]]

def extract_segments_for_class(dfin, cls):
    g = dfin[dfin[cls] >= PROBA_THR]
    if g.empty:
        return pd.DataFrame(columns=["video_id","agent_id","action","start_frame","stop_frame"])

    g = g.sort_values(["video_id","agent_id","video_frame"])
    out = []

    for (vid, aid), sub in g.groupby(["video_id","agent_id"]):
        sub = sub.sort_values("video_frame")
        # New segment if frames are not consecutive
        seg_id = (sub["video_frame"].diff().ne(1)).cumsum()
        segs = (sub.groupby(seg_id)
                    .agg(start_frame=("video_frame","min"),
                         stop_frame=("video_frame","max"))
                    .reset_index(drop=True))
        # filter by length
        segs["length"] = segs["stop_frame"] - segs["start_frame"] + 1
        segs = segs[segs["length"] >= MIN_FRAMES]
        if not segs.empty:
            segs.insert(0, "action", cls)
            segs.insert(0, "agent_id", int(aid))
            segs.insert(0, "video_id", vid)
            out.append(segs[["video_id","agent_id","action","start_frame","stop_frame"]])

    return pd.concat(out, ignore_index=True) if out else pd.DataFrame(columns=["video_id","agent_id","action","start_frame","stop_frame"])

def sanity_check_submission(submission):
    if len(submission) == 0:
        return pd.read_csv(INPUT_DIRECTORY / "sample_submission.csv").astype({"video_id":"Int64"})
        
    # remove actions with invalid start_frame/stop_frame combinations
    submission = submission[submission.start_frame < submission.stop_frame]
    
    # remove overlapping actions per agent/target pair
    group_list = []
    for _, group in submission.groupby(['video_id', 'agent_id', 'target_id']):
        group = group.sort_values('start_frame')
        mask = np.ones(len(group), dtype=bool)
        last_stop_frame = 0
        for i, (_, row) in enumerate(group.iterrows()):
            if row['start_frame'] < last_stop_frame:
                mask[i] = False
            else:
                last_stop_frame = row['stop_frame']
        group_list.append(group[mask])
    submission = pd.concat(group_list)

    # order by start_frame and make sure row_id is monotonic and without duplicates
    submission = submission.sort_values(by="start_frame")
    submission["row_id"] = range(len(submission))

    # remove untracked actions
    submission["behavior"] = submission["agent_id"].astype(str) + "," + submission["target_id"].astype(str) + "," + submission["action"]
    submission["behavior_tracked"] = submission.apply(
        lambda row: row["behavior"] in allowed_behaviors_by_video.get(row["video_id"], []),
        axis=1
    )
    return submission

def prediction2submission(preds: np.ndarray, meta: pd.DataFrame, classes: list[str]) -> pd.DataFrame:
    # Combine meta + per-class probabilities (columns aligned to global CLASSES)
    df = meta.copy().reset_index(drop=True)
    proba_df = pd.DataFrame(preds, columns=classes)
    df = pd.concat([df, proba_df], axis=1)

    segments = pd.concat(
        [extract_segments_for_class(df, cls) for cls in classes if cls != "no_action"],
        ignore_index=True
    )
    df = transform_ego_predictions(segments)
    submission = sanity_check_submission(df)
    return submission

## **INGESTION AND PREPROCESSING**

### Read metadata, encode bodyparts to more broad body regions, encode tracked behaviours in one-hot fashion

In [25]:
train_metadata = pd.read_csv(INPUT_DIRECTORY / "train.csv")
train_metadata = train_metadata[train_metadata["lab_id"].isin(LABS_CONSIDERED)]
test_metadata = pd.read_csv(INPUT_DIRECTORY / "test.csv")
test_metadata = test_metadata[test_metadata["lab_id"].isin(LABS_CONSIDERED)]
metadata = pd.concat([train_metadata, test_metadata])

print(f"read {len(train_metadata)} train rows.")
print(f"read {len(test_metadata)} test rows.")

read 17 train rows.
read 1 test rows.


In [26]:
# Determine the tracked bodyparts of each video
behaviors = pd.concat([
    train_metadata[["video_id", "behaviors_labeled"]],
    test_metadata[["video_id", "behaviors_labeled"]]
])
behaviors = behaviors.dropna()

# Convert the string to a list using ast.literal_eval
behaviors.loc[:, "behaviors_labeled"] = behaviors.loc[:, "behaviors_labeled"]#.apply(ast.literal_eval)

# Convert to dictionary that returns a list of tracked behaviors per video id
allowed_behaviors_by_video = behaviors.set_index("video_id").to_dict()["behaviors_labeled"]

behaviors_json_by_video = {}
for k, v in allowed_behaviors_by_video.items():
    try:
        vid = int(k)
    except Exception:
        # if k is already Int64, coerce via astype below; keep string fallback
        vid = int(str(k))
    # v might already be a list or a string-repr of a list
    if isinstance(v, list):
        behaviors_json_by_video[vid] = json.dumps(v)
    else:
        # try to parse string-repr to list, otherwise store as-is if already JSON
        try:
            parsed = ast.literal_eval(v)
            behaviors_json_by_video[vid] = json.dumps(parsed if isinstance(parsed, list) else [])
        except Exception:
            # final fallback: if it's already a JSON string, keep; else store empty list
            try:
                json.loads(v)
                behaviors_json_by_video[vid] = v
            except Exception:
                behaviors_json_by_video[vid] = "[]"

In [27]:
# Determine the tracked bodyparts of each video
bodyparts = train_metadata[["lab_id", "video_id", "body_parts_tracked"]]

# Convert the string to a list using ast.literal_eval
bodyparts.loc[:, "body_parts_tracked"] = bodyparts.loc[:, "body_parts_tracked"].apply(ast.literal_eval)

# Encode using MultiLabelBinarizer
mlb = MultiLabelBinarizer()
bodyparts_encoded = pd.DataFrame(
    mlb.fit_transform(bodyparts["body_parts_tracked"]),
    columns=mlb.classes_,
    index=bodyparts.index
)

# Confirm everything worked as expected
print(f"Example row for lab_id {bodyparts.iloc[0]['lab_id']} and video_id {bodyparts.iloc[0]['video_id']}:\n\n")
print(f"Original bodyparts literal:\n{bodyparts.iloc[0]['body_parts_tracked']}\n\n")
print(f"Encoded one-hot vector:\n{bodyparts_encoded.iloc[0]}")

Example row for lab_id AdaptableSnail and video_id 44566106:


Original bodyparts literal:
['body_center', 'ear_left', 'ear_right', 'headpiece_bottombackleft', 'headpiece_bottombackright', 'headpiece_bottomfrontleft', 'headpiece_bottomfrontright', 'headpiece_topbackleft', 'headpiece_topbackright', 'headpiece_topfrontleft', 'headpiece_topfrontright', 'lateral_left', 'lateral_right', 'neck', 'nose', 'tail_base', 'tail_midpoint', 'tail_tip']


Encoded one-hot vector:
body_center                   1
ear_left                      1
ear_right                     1
headpiece_bottombackleft      1
headpiece_bottombackright     1
headpiece_bottomfrontleft     1
headpiece_bottomfrontright    1
headpiece_topbackleft         1
headpiece_topbackright        1
headpiece_topfrontleft        1
headpiece_topfrontright       1
lateral_left                  1
lateral_right                 1
neck                          1
nose                          1
tail_base                     1
tail_midpoint      

In [28]:
# Map bodyparts to larger body regions in order to 
# be able to unify the data better at a later stage
# since each video tracks different body parts and 
# not all video frames include data points for all
# tracked bodyparts

bodypart_to_region = {
    # Head region
    "ear_left": "head", "ear_right": "head", "nose": "head", "head": "head", "neck": "head", "headpiece_bottombackleft": "head", "headpiece_topfrontright": "head", "headpiece_bottomfrontleft": "head", "headpiece_topfrontleft": "head", "headpiece_topbackright": "head", "headpiece_topbackleft": "head", "headpiece_bottomfrontright": "head", "headpiece_bottombackright": "head",
    # Torso
    "body_center": "torso", "spine_1": "torso","spine_2": "torso","lateral_right": "torso","lateral_left": "torso","hip_left": "torso","hip_right": "torso",
    # Limbs
    "forepaw_left": "limbs_front","forepaw_right": "limbs_front","hindpaw_left": "limbs_hind","hindpaw_right": "limbs_hind",    
    # Tail
    "tail_base": "tail","tail_tip": "tail","tail_midpoint": "tail","tail_middle_1": "tail","tail_middle_2": "tail",
}

In [29]:
# tracking data
train_tracking_df = ingest_tracking_data("train")
test_tracking_df = ingest_tracking_data("test")

# annotation data
train_annotation_df = ingest_annotation_data("train")

reading /Users/hs/dev/TabularShenanigans/kaggle/input/MABe-mouse-behavior-detection/train_tracking/AdaptableSnail/705948978.parquet
reading /Users/hs/dev/TabularShenanigans/kaggle/input/MABe-mouse-behavior-detection/train_tracking/AdaptableSnail/355542626.parquet
reading /Users/hs/dev/TabularShenanigans/kaggle/input/MABe-mouse-behavior-detection/train_tracking/AdaptableSnail/2078515636.parquet
reading /Users/hs/dev/TabularShenanigans/kaggle/input/MABe-mouse-behavior-detection/train_tracking/AdaptableSnail/1596473327.parquet
reading /Users/hs/dev/TabularShenanigans/kaggle/input/MABe-mouse-behavior-detection/train_tracking/AdaptableSnail/1212811043.parquet
reading /Users/hs/dev/TabularShenanigans/kaggle/input/MABe-mouse-behavior-detection/train_tracking/AdaptableSnail/351967631.parquet
reading /Users/hs/dev/TabularShenanigans/kaggle/input/MABe-mouse-behavior-detection/train_tracking/AdaptableSnail/1408652858.parquet
reading /Users/hs/dev/TabularShenanigans/kaggle/input/MABe-mouse-behavio

KeyError: "None of [Index(['1212811043', '1212811043', '1212811043', '1212811043', '1212811043',\n       '1212811043', '1212811043', '1212811043', '1212811043', '1212811043',\n       ...\n       '878123481', '878123481', '878123481', '878123481', '878123481',\n       '878123481', '878123481', '878123481', '878123481', '878123481'],\n      dtype='object', length=2192098)] are in the [index]"

## **TRANSFORMATIONS**

Every frame in the tracking data will be used from each mouse's perspective once. The following section generates transformations between the general and ego perspectives.

In [ ]:
meta_columns = ["video_id", "video_frame", "agent_id"]

# transform tracking and annotation data to X and y
train_data = pd.concat([tracking_to_ego_df(train_tracking_df, ego) for ego in range(1, 5, 1)], ignore_index=True)
train_labels = pd.concat([annotation_to_ego_df(train_annotation_df, ego) for ego in range(1, 5, 1)], ignore_index=True)
test_data = pd.concat([tracking_to_ego_df(test_tracking_df, ego) for ego in range(1, 5, 1)], ignore_index=True)

# align X_train and y_train on video_id, video_frame and agent_id 
train_data = train_data.merge(train_labels, how="left", on=meta_columns)
train_data["label"] = train_data["label"].fillna("no_action")

# split into meta, X and y dataframes
meta_train = train_data[meta_columns]
X_train = train_data.drop(columns=meta_columns + ["label"])
CLASSES: list[str] = train_data["label"].astype("category").cat.categories.tolist()
label2id = {c: i for i, c in enumerate(CLASSES)}
id2label = {i: c for c, i in label2id.items()}
y_train = train_data["label"].map(label2id).astype("int32")
N_CLASSES = len(CLASSES)

X_test = test_data.drop(columns=meta_columns)
meta_test = test_data[meta_columns]

## CROSS VALIDATION

In [ ]:
def fit_and_predict_fn(X_train, y_train_int, X_val):
    from xgboost import XGBClassifier
    model = XGBClassifier(random_state=RANDOM_STATE)
    model.fit(X_train, y_train_int)
    proba = model.predict_proba(X_val)
    # Align to global class space [0..N_CLASSES-1]
    proba = align_proba_to_global(proba, model.classes_, N_CLASSES)
    return proba
    
def align_proba_to_global(proba: np.ndarray,
                          model_classes: np.ndarray,
                          n_classes: int) -> np.ndarray:
    """
    Expand model proba (n, k_fold) to (n, n_classes) using zeros
    for classes unseen in this fold. model_classes are int-coded labels.
    """
    out = np.zeros((proba.shape[0], n_classes), dtype=float)
    out[:, model_classes] = proba
    return out

def ints_to_onehot_df(y_int: pd.Series, n_classes: int, columns: list[str]) -> pd.DataFrame:
    ohe = np.zeros((len(y_int), n_classes), dtype=float)
    ohe[np.arange(len(y_int)), y_int.to_numpy()] = 1.0
    return pd.DataFrame(ohe, columns=columns, index=y_int.index)
    
def run_cv_with_mabe_metric(X, y, meta, n_splits, fit_and_predict_fn):
    vids = meta[["video_id"]].values
    gkf = GroupKFold(n_splits=n_splits)

    fold_scores = []
    for fold, (tr, va) in enumerate(gkf.split(X=X, y=y, groups=vids), 1):
        print(f"training data split into {len(tr)} training and {len(va)} validation examples.")
        X_train, y_train_int, meta_train = X.iloc[tr], y.iloc[tr], meta.iloc[tr]
        X_val,   y_val_int,   meta_val   = X.iloc[va], y.iloc[va], meta.iloc[va]

        # model predictions -> aligned to global CLASSES
        y_pred = fit_and_predict_fn(X_train, y_train_int, X_val)
        y_pred_sub = prediction2submission(y_pred, meta_val, CLASSES)

        # ground truth for scoring: turn int labels into one-hot probs aligned to CLASSES
        y_val_ohe = ints_to_onehot_df(y_val_int, N_CLASSES, CLASSES).to_numpy()
        y_val_sub = prediction2submission(y_val_ohe, meta_val, CLASSES)

        # Ensure video_id is int for mapping (adjust if you keep them as strings everywhere)
        y_val_sub["video_id"] = pd.to_numeric(y_val_sub["video_id"], errors="coerce").astype("Int64")
        y_pred_sub["video_id"] = pd.to_numeric(y_pred_sub["video_id"], errors="coerce").astype("Int64")
        
        # Required by scorer
        # Attach JSON strings; fill any misses with "[]"
        y_val_sub["behaviors_labeled"] = (
            y_val_sub["video_id"].astype("Int64").map(behaviors_json_by_video).fillna("[]")
        )
        y_val_sub["lab_id"] = 1
        s = score(solution=y_val_sub, submission=y_pred_sub, row_id_column_name="row_id")
        fold_scores.append({"fold": fold, "score": s})

    df = pd.DataFrame(fold_scores)
    return df, float(np.mean(df["score"]))


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

scores, mean_score = run_cv_with_mabe_metric(
    X=X_train,
    y=y_train,
    meta=meta_train,
    n_splits=N_CV_SPLITS,
    fit_and_predict_fn=fit_and_predict_fn
)

In [ ]:
scores

In [ ]:
mean_score

### **LEADERBOARD**:

BASELINE XGB with 7-fold GKF and only pos/dir features: CV 0.026429026596703892, LB 0.0